In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timedelta
import random

In [0]:
# 🏗️ Setup: SwissLogistics Data Platform## Practice Project — Data Generation**Scenario:** SwissLogistics AG is a mid-size logistics company based in Zurich. They operate a fleet of 200+ delivery vehicles across Switzerland, handle ~50K shipments/month, and serve both B2B and B2C customers.**Your role:** You're building their data platform on Databricks. The company has multiple data sources:- **Shipment events** — GPS tracking pings, status updates (picked up, in transit, delivered, failed)- **Customer/partner data** — company profiles that change over time (addresses, tiers, contacts)- **Orders** — order lifecycle with payments, cancellations, returns- **Vehicle telemetry** — sensor data from delivery vehicles (speed, fuel, temperature for cold-chain)**Run this notebook first** to generate all source data. Then work through notebooks 01-08 in order.---### Setup Instructions1. Import all notebooks into a Databricks workspace (Free Edition works)2. Run this notebook to generate source data3. Work through each notebook — they build on each other

In [0]:
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
print(configs)

{'env': 'dev'}


In [0]:
# ── Configs ──
ENV = configs.get('env', 'dev')

CATALOG = f"sl_{ENV}"
SCHEMA_INGEST = ENV
SCHEMA_BRONZE = "bronze"  
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
LANDING_VOL = "landing"
LANDING_BASE = f"/Volumes/sl_ingest/{SCHEMA_INGEST}/{LANDING_VOL}"

In [0]:
# Clean up workspace
for catalog in [CATALOG, "sl_ingest"]:
    spark.sql(f"DROP CATALOG IF EXISTS {catalog} CASCADE")
    
display(spark.sql("SHOW CATALOGS"))

catalog
databricks_simulated_e_commerce_clickstream_data
databricks_simulated_retail_customer_data
samples
system
workspace


In [0]:
# Create catalogs
for catalog in [CATALOG, "sl_ingest"]:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
    
display(spark.sql("SHOW CATALOGS"))

catalog
databricks_simulated_e_commerce_clickstream_data
databricks_simulated_retail_customer_data
samples
sl_dev
sl_ingest
system
workspace


In [0]:


# Create schemas in env catalog (use default catalog on Community Edition)
for schema in [SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{schema}.checkpoints")
    
display(spark.sql("SHOW SCHEMAS IN workspace"))


databaseName
de_pro_practice
default
information_schema
sl_bronze
sl_gold
sl_raw
sl_silver


In [0]:
#the rest of this notebook uses the ingest catalog
spark.sql("USE CATALOG sl_ingest")

DataFrame[]

In [0]:
#create schema and volume in sl_ingest catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_INGEST}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA_INGEST}.{LANDING_VOL}")

DataFrame[]

## Generate Source Data

In [0]:
random.seed(42)
BASE_DATE = datetime(2025, 1, 1)

# ── CUSTOMERS (200 companies) ──
# Simulates the operational Postgres "customers" table: CURRENT STATE ONLY,
# one row per customer. No SCD2 bookkeeping lives here (no is_current, no version
# history). signup_date = created_at, last_updated = last modification time.
# The warehouse builds the Type 2 dimension downstream from this table's CDC feed.
swiss_cities = ["Zürich", "Bern", "Basel", "Genève", "Lausanne", "Winterthur",
                "St. Gallen", "Luzern", "Biel", "Thun", "Aarau", "Schaffhausen",
                "Lugano", "Fribourg", "Chur", "Neuchâtel", "Zug", "Sion"]
tiers = ["standard", "premium", "enterprise"]
industries = ["pharma", "retail", "manufacturing", "food_beverage", "tech", "finance"]

customer_records = []
for cid in range(1, 201):
    city = random.choice(swiss_cities)
    tier = random.choices(tiers, weights=[50, 35, 15])[0]
    industry = random.choice(industries)
    signup = BASE_DATE - timedelta(days=random.randint(60, 1500))
    # Most rows were last touched at signup; ~30% had an in-system edit since then.
    last_updated = signup + timedelta(days=random.randint(30, 365)) if random.random() < 0.3 else signup
    customer_records.append((
        cid, f"Company_{cid:04d}", f"contact_{cid}@company{cid}.ch",
        f"{random.randint(1,200)} {random.choice(['Bahnhofstrasse','Seestrasse','Hauptstrasse','Industriestrasse'])}",
        city, f"{random.randint(1000,9999)}", tier, industry, signup, last_updated
    ))

cust_schema = StructType([
    StructField("customer_id", IntegerType()), StructField("company_name", StringType()),
    StructField("contact_email", StringType()), StructField("address", StringType()),
    StructField("city", StringType()), StructField("postal_code", StringType()),
    StructField("tier", StringType()), StructField("industry", StringType()),
    StructField("signup_date", TimestampType()), StructField("last_updated", TimestampType())
])

df_customers = spark.createDataFrame(customer_records, cust_schema)
df_customers.write.mode("overwrite").option("delta.enableChangeDataFeed", "true").saveAsTable(f"{SCHEMA_INGEST}.customers")
print(f"✓ customers: {df_customers.count()} current-state rows (one per customer, Postgres-style)")

✓ customers: 270 records (200 current, rest historical)


In [0]:
# ── ORDERS (50K orders over 12 months) ──
products = [
    ("PKG_STD", "Standard Parcel", 12.50), ("PKG_EXP", "Express Parcel", 24.90),
    ("PKG_SAME", "Same-Day Delivery", 39.90), ("FRT_PAL", "Pallet Freight", 89.00),
    ("FRT_FTL", "Full Truck Load", 450.00), ("COLD", "Cold Chain Parcel", 34.90),
    ("DOC", "Document Courier", 8.90), ("INTL", "International Shipment", 65.00),
]
statuses = ["created", "confirmed", "picked_up", "in_transit", "out_for_delivery", 
            "delivered", "failed_delivery", "returned"]
payment_methods = ["invoice", "credit_card", "bank_transfer", "paypal"]

order_records = []
for oid in range(1, 50001):
    cid = random.randint(1, 200)
    prod = random.choice(products)
    qty = random.randint(1, 10) if prod[0].startswith("PKG") else random.randint(1, 3)
    order_date = BASE_DATE + timedelta(days=random.randint(0, 364), hours=random.randint(6, 22),
                                        minutes=random.randint(0, 59))
    status_idx = random.choices(range(len(statuses)), weights=[5,5,8,15,10,45,7,5])[0]
    status = statuses[status_idx]
    
    # Delivery date only if delivered
    delivery_date = order_date + timedelta(days=random.randint(1, 5)) if status in ("delivered", "returned") else None
    
    # Some orders have NULL fields (data quality issues for silver layer)
    amount = __builtins__.round(prod[2] * qty * random.uniform(0.9, 1.1), 2)
    if random.random() < 0.02: 
        amount = None  # ~2% missing amounts
    
    order_records.append((
        f"ORD-{oid:06d}", cid, prod[0], prod[1], qty, amount,
        status, random.choice(payment_methods), order_date, delivery_date,
        random.choice(swiss_cities),  # origin
        random.choice(swiss_cities),  # destination
    ))

order_schema = StructType([
    StructField("order_id", StringType()), StructField("customer_id", IntegerType()),
    StructField("product_code", StringType()), StructField("product_name", StringType()),
    StructField("quantity", IntegerType()), StructField("total_amount", DoubleType(), True),
    StructField("status", StringType()), StructField("payment_method", StringType()),
    StructField("order_date", TimestampType()), StructField("delivery_date", TimestampType(), True),
    StructField("origin_city", StringType()), StructField("destination_city", StringType()),
])

df_orders = spark.createDataFrame(order_records, order_schema)
df_orders.write.mode("overwrite").option("delta.enableChangeDataFeed", "true").saveAsTable(f"{SCHEMA_INGEST}.orders")
print(f"✓ orders: {df_orders.count()} records")

✓ orders: 50000 records


In [0]:
# ── SHIPMENT EVENTS (GPS tracking + status updates, ~500K events) ──
# This is the high-volume streaming-like data

event_types = ["gps_ping", "status_change", "temperature_alert", "delay_alert", "signature_captured"]

shipment_events = []
for i in range(500000):
    oid = f"ORD-{random.randint(1, 50000):06d}"
    vehicle = f"VH-{random.randint(1, 200):03d}"
    evt_type = random.choices(event_types, weights=[60, 20, 5, 10, 5])[0]
    ts = BASE_DATE + timedelta(days=random.randint(0, 364), hours=random.randint(0, 23),
                                minutes=random.randint(0, 59), seconds=random.randint(0, 59))
    
    # GPS coordinates (roughly Switzerland: lat 46-48, lon 6-10)
    lat = __builtins__.round(random.uniform(46.0, 47.8), 6) if evt_type == "gps_ping" else None
    lon = __builtins__.round(random.uniform(6.0, 10.5), 6) if evt_type == "gps_ping" else None
    
    # Status for status_change events
    status = random.choice(statuses) if evt_type == "status_change" else None
    
    # Temperature for cold chain alerts
    temp = __builtins__.round(random.gauss(4, 2), 1) if evt_type == "temperature_alert" else None
    
    # Inject duplicates (~3% — common in real event streams)
    shipment_events.append((
        f"EVT-{i+1:07d}", oid, vehicle, evt_type, ts, lat, lon, status, temp,
        random.choice(["kafka", "iot_hub", "api"]),  # source system
    ))
    if random.random() < 0.03:
        shipment_events.append((
            f"EVT-{i+1:07d}", oid, vehicle, evt_type, ts, lat, lon, status, temp,
            random.choice(["kafka", "iot_hub", "api"]),
        ))

event_schema = StructType([
    StructField("event_id", StringType()), StructField("order_id", StringType()),
    StructField("vehicle_id", StringType()), StructField("event_type", StringType()),
    StructField("event_timestamp", TimestampType()),
    StructField("latitude", DoubleType(), True), StructField("longitude", DoubleType(), True),
    StructField("shipment_status", StringType(), True),
    StructField("temperature_celsius", DoubleType(), True),
    StructField("source_system", StringType()),
])

df_events = spark.createDataFrame(shipment_events, event_schema)
(df_events.repartition(100)  # ~5K/file, ~1-2 MB
   .write.mode("overwrite")  # initial seed only
   .json(f"{LANDING_BASE}/shipment_events"))

In [0]:
df_events_validate = spark.read.json(f"{LANDING_BASE}/shipment_events/")
print(f"✓ shipment_events: {df_events_validate.count()} records (includes ~3% intentional duplicates)")

✓ shipment_events: 515045 records (includes ~3% intentional duplicates)


In [0]:
import uuid
# ── VEHICLE TELEMETRY (sensor readings, ~200K rows, SKEWED by vehicle) ──
# vehicle VH-001 to VH-005 generate 60% of all data (skew for optimization practice)

telemetry = []
for i in range(200000):
    # Intentional skew: 60% of readings from 5 vehicles
    if random.random() < 0.6:
        vehicle = f"VH-{random.randint(1, 5):03d}"
    else:
        vehicle = f"VH-{random.randint(6, 200):03d}"
    
    ts = BASE_DATE + timedelta(days=random.randint(0, 364), hours=random.randint(0, 23),
                                minutes=random.randint(0, 59), seconds=random.randint(0, 59))
    telemetry.append((
        str(uuid.uuid4()),                          # reading_id (telematics gateway message id)
        vehicle, ts,
        __builtins__.round(random.uniform(0, 120), 1),       # speed_kmh
        __builtins__.round(random.uniform(0, 100), 1),        # fuel_pct
        __builtins__.round(random.gauss(22, 8), 1),           # engine_temp_c
        __builtins__.round(random.gauss(4, 3), 1) if random.random() < 0.3 else None,  # cargo_temp (only cold chain)
        random.randint(0, 500000),               # odometer_km
        random.choice(["idle", "moving", "loading", "maintenance"]),
    ))

tel_schema = StructType([
    StructField("reading_id", StringType()),
    StructField("vehicle_id", StringType()), StructField("reading_timestamp", TimestampType()),
    StructField("speed_kmh", DoubleType()), StructField("fuel_pct", DoubleType()),
    StructField("engine_temp_c", DoubleType()), StructField("cargo_temp_c", DoubleType(), True),
    StructField("odometer_km", IntegerType()), StructField("vehicle_status", StringType()),
])

df_telemetry = spark.createDataFrame(telemetry, tel_schema)
(df_telemetry.repartition(40)  # ~5K/file, ~1 MB
   .write.mode("overwrite")  # initial seed only
   .json(f"{LANDING_BASE}/telemetry"))

In [0]:
df_telemetry_validate = spark.read.json(f"{LANDING_BASE}/telemetry/")
print(f"✓ vehicle_telemetry: {df_telemetry_validate.count()} records (intentionally skewed — 60% from 5 vehicles)")


✓ vehicle_telemetry: 200000 records (intentionally skewed — 60% from 5 vehicles)


In [0]:
# ── INCOMING CDC BATCH (for MERGE/upsert exercises) ──
# Simulates a daily CDC feed: new orders + updates to existing orders

cdc_records = []
# 500 updates to existing orders (status changes)
for i in range(500):
    oid = f"ORD-{random.randint(1, 50000):06d}"
    new_status = random.choice(["delivered", "failed_delivery", "returned"])
    cdc_records.append((oid, None, None, None, None, None, new_status, None,
                        None, BASE_DATE + timedelta(days=365), None, None, "UPDATE"))

# 200 new orders
for i in range(200):
    oid = f"ORD-{50000 + i + 1:06d}"
    cid = random.randint(1, 200)
    prod = random.choice(products)
    qty = random.randint(1, 5)
    cdc_records.append((
        oid, cid, prod[0], prod[1], qty, __builtins__.round(prod[2] * qty, 2),
        "created", random.choice(payment_methods),
        BASE_DATE + timedelta(days=365), None,
        random.choice(swiss_cities), random.choice(swiss_cities), "INSERT"
    ))

cdc_schema = StructType([
    StructField("order_id", StringType()), StructField("customer_id", IntegerType(), True),
    StructField("product_code", StringType(), True), StructField("product_name", StringType(), True),
    StructField("quantity", IntegerType(), True), StructField("total_amount", DoubleType(), True),
    StructField("status", StringType()), StructField("payment_method", StringType(), True),
    StructField("order_date", TimestampType(), True), StructField("delivery_date", TimestampType(), True),
    StructField("origin_city", StringType(), True), StructField("destination_city", StringType(), True),
    StructField("_change_type", StringType()),
])

df_cdc = spark.createDataFrame(cdc_records, cdc_schema)
(df_cdc.coalesce(1)  # single file (it represents one daily batch)
   .write.mode("overwrite")  # initial seed only
   .json(f"{LANDING_BASE}/cdc_records"))

In [0]:
df_cdc_validate = spark.read.json(f"{LANDING_BASE}/telemetry/")
print(f"✓ orders_cdc_batch: {df_cdc_validate.count()} records (500 updates + 200 inserts)")

✓ orders_cdc_batch: 200000 records (500 updates + 200 inserts)


In [0]:
# ── SUMMARY ──
print("=" * 60)
print("SwissLogistics Data Platform — Source Data Ready")
print("=" * 60)
for table in ["customers", "orders", "shipment_events", "telemetry", "cdc_records"]:
    if table in ["customers", "orders"]:
        count = spark.table(f"{SCHEMA_INGEST}.{table}").count()
        print(f"  {SCHEMA_INGEST}.{table}: {count:,} rows")
    else:
        files = dbutils.fs.ls(f"{LANDING_BASE}/{table}/")
        json_files = [f for f in files if f.name.lower().endswith(".json")]
        print(f"  {LANDING_BASE}/{table}: {len(json_files)} files")
print()
print("Data characteristics:")
print("  • customers: current-state source (Postgres-style), CDC feeds the SCD2 build")
print("  • orders: ~2% NULL amounts (data quality issue)")
print("  • shipment_events: ~3% duplicates (dedup practice)")
print("  • vehicle_telemetry: 60% skewed to 5 vehicles (skew handling)")
print("  • orders_cdc_batch: CDC feed for MERGE/upsert practice")
print()
print("Next: Open 01_Bronze_Ingestion.ipynb")

SwissLogistics Data Platform — Source Data Ready
  dev.customers: 270 rows
  dev.orders: 50,000 rows
  /Volumes/sl_ingest/dev/landing/shipment_events: 100 files
  /Volumes/sl_ingest/dev/landing/telemetry: 40 files
  /Volumes/sl_ingest/dev/landing/cdc_records: 1 files

Data characteristics:
  • customers: includes historical records for SCD2 practice
  • orders: ~2% NULL amounts (data quality issue)
  • shipment_events: ~3% duplicates (dedup practice)
  • vehicle_telemetry: 60% skewed to 5 vehicles (skew handling)
  • orders_cdc_batch: CDC feed for MERGE/upsert practice

Next: Open 01_Bronze_Ingestion.ipynb
